# Part 1 — Why RAG Exists

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-01-why-rag/why-rag.ipynb)

*A capable language model still answers confidently and wrong about your own documents and about anything recent. This part is the **why** behind every line of code in the rest of the series.*

📖 **Read the essay:** https://www.mefby.com/essays/why-rag-exists

---

**What this notebook covers**

- The four limitations of a vanilla LLM: hallucination, knowledge cutoff, no private knowledge, context-window limits.
- The fix in three moves: **R**etrieve → **A**ugment → **G**enerate.
- One durable mental model — the open-book exam — and what it predicts about where RAG fails.
- The two-phase pipeline: index once, query per question.
- When to reach for RAG versus a bigger prompt versus fine-tuning.

> **No code here.** This is a conceptual chapter — the intuition the whole series rests on. The first line of runnable code arrives in **Part 2 — Embeddings**.

## The confidently wrong answer

Picture a moment you may have lived through. You ask a genuinely capable model — the kind that writes working code and drafts your emails — something that matters to *your* world:

> *"What's our company's refund window for orders placed in 2026?"*

It answers instantly, in fluent, self-assured prose: *"Refunds are accepted within 90 days of purchase."* Clean. Confident. **Wrong** — your window is 30 days, and the number 90 appears nowhere in your handbook. The model never read your handbook (it has never seen it), so it did the only thing it knows how to do: produce text that *sounds* like a refund policy.

Now ask: *"Summarize the incident our team filed yesterday."* Same fluency, same confidence — but this time yesterday hadn't happened yet when the model was built. It *cannot* know.

Neither failure is a bug you can prompt your way out of. They are **structural**. On its own, a language model is a brilliant reasoner with **no access to your information and no clock**. Before we fix that, we need to be precise about *why* it happens — because each cause maps to a thing RAG does.

## Why a vanilla LLM gets it wrong: four limitations

First, the thing we're critiquing. A **large language model (LLM)** is a model trained on an enormous pile of text to do one deceptively simple task: *given some text, predict the next chunk of text.* That's it. Everything it appears to "know" is a side effect of getting very, very good at that prediction game across a huge fraction of the internet.

That single design choice produces four limitations. They aren't random flaws; they fall out directly from "predict the next piece of text." We'll take them one at a time.

### 1. Hallucination

A **hallucination** is when the model states something false *with the same confidence it states something true* — a fabricated fact delivered in a steady voice, not a typo or a hedge.

**Why it happens:** the model was never trained to track *truth*. It was trained to produce *likely text*. Asked for your refund window, it isn't consulting a fact and reporting it; it's generating the most probable continuation of "our refund window is ___." If thousands of refund policies in its training said "90 days," then "90 days" is a *probable* continuation — and probable is all it optimizes for. There is no internal step where it checks "is this correct for *this* company?", because it has no ground truth to check against and no copy of your policy to check it in.

The uncomfortable part: the model often sounds *most* confident exactly when it's making things up, because fluent invention and fluent recall come from the same machinery. **Confidence is not evidence.**

### 2. Knowledge cutoff

Every LLM is built from a snapshot of text collected *up to a certain date*. That date is its **knowledge cutoff**: the point where its training data ends and, as far as the model is concerned, history stops.

Ask about anything after that line — yesterday's incident, this morning's release, a law passed last week — and it simply has no data. It will still answer (it always answers), but it's now extrapolating into a void. The frozen snapshot is also why a model can't "learn" your correction mid-conversation in any lasting way: its weights were fixed at training time while the world kept moving without it.

### 3. No private or proprietary knowledge

The knowledge cutoff is about *when*. This one is about *what*. Even for events well before the cutoff, the model only ever saw **public** text — roughly, what was scrapeable from the open internet plus licensed datasets.

It never saw your internal wiki. Not your Slack, not your design docs, not your customer database, not the PDF that actually governs refunds. None of it was in the training pile, so none of it is in the model. This is usually the limitation that bites businesses hardest: the single most valuable knowledge you have — *your* knowledge — is precisely what a general-purpose model is guaranteed not to possess. A bigger or smarter model doesn't help, because the issue isn't capability; **the information was never in the room.**

### 4. Context-window limits

"Fine," you might say, "I'll just paste our whole handbook into the prompt every time." Right instinct — and it runs straight into a wall. Two terms explain the wall.

A **token** is the unit a model reads and writes in: a chunk of text, very roughly ¾ of an English word, so "refund" might be one token and "unbelievable" three. Models see sequences of tokens, not letters or words.

The **context window** is the maximum number of tokens the model can hold in mind at once — your prompt *and* its answer combined. Think of it as the model's working memory, or the size of its desk. It's finite and fixed for a given model.

So pasting "everything" fails for concrete reasons:

- **It doesn't fit.** A modest knowledge base is millions of tokens; a context window holds a few hundred thousand at most. Most of your data physically can't be in the prompt at once.
- **It's wasteful.** Every token you send costs money and latency on *every* request. Shipping the entire handbook to answer one refund question is like re-reading the whole binder aloud before each sentence.
- **It degrades.** Bury one relevant line among a hundred thousand irrelevant ones and models get measurably worse at finding it; the signal drowns. More context is not the same as more *relevant* context.

The fix isn't a bigger desk. It's putting *only the right page* on the desk at the right moment — which is exactly what RAG does.

## The fix: Retrieve → Augment → Generate

Here's the whole idea in one sentence:

> **RAG fetches the most relevant pieces of your own data and places them into the model's prompt, so the model answers from *evidence you supplied* instead of from memory alone.**

We're not retraining the model, not changing its weights, not teaching it anything permanent. We're changing *what it sees at the moment it answers*. The name spells out the three moves, in order:

- **R — Retrieval.** Search your knowledge for the pieces most relevant to the question and pull them out. Asked about refunds, retrieval finds the actual paragraph from your actual policy. This is what supplies private data **and** anything newer than the cutoff (fixes limitations 3 and 2).
- **A — Augmentation.** Insert the retrieved pieces into the prompt alongside the user's question, usually with a quiet instruction like "answer using the context below." We put *only the relevant slice* on the desk, sidestepping the context-window wall (fixes limitation 4).
- **G — Generation.** The model does what it's genuinely great at — writing a fluent, coherent answer — except now grounded in the supplied evidence rather than spun from probability. Because the answer is anchored to real text it can read and even cite, hallucination drops sharply (fixes limitation 1).

Notice how cleanly three moves answer four problems. Same talented writer, finally given the right source material.

## The mental model: an open-book exam

If you remember one image from this whole part, make it this one. Picture two students taking the *same* exam.

The first sits a **closed-book** exam. When a question lands outside what she happens to remember, she can't look anything up, so she writes her best guess in confident handwriting — a blank scores zero, a plausible answer might not. Sometimes she's right; sometimes she invents a citation that doesn't exist. She can't tell those two cases apart, and neither can you from the paper alone.

The second sits the **open-book** version of the *exact same exam*. Same student, same brain, same skill — but now she may bring the textbook. When a question lands, she flips to the relevant page, lays the book open beside the question, reads the passage, and writes the answer in her own words.

The closed-book student is a **vanilla LLM**. The open-book student is **RAG**. *Nothing about the student's intelligence changed — only her access to the source did.* The analogy maps one-to-one onto the mechanism:

- Flipping to the right page **= retrieval** (find the relevant material)
- Laying the open book beside the question **= augmentation** (put that material in front of the model with the question)
- Writing the answer in her own words **= generation** (the model composes the response from what's there)

The analogy even predicts RAG's **failure modes**, which is how you know it's a good one. If the textbook doesn't contain the answer, the open-book student is no better off — RAG can only ground answers in knowledge you actually gave it. And if she flips to the *wrong* page, she'll confidently answer from irrelevant material. That's why **retrieval quality is the whole ballgame**, and why much of this series is about getting the right page into her hands.

## A bird's-eye view: the two-phase pipeline

How do we let a program "flip to the right page" across thousands of documents in milliseconds? You don't read everything at question time — that's the context-window wall again. Instead you do a little preparation *ahead* of time so that, at question time, finding the right page is fast.

There are two phases. The first happens **once**, up front, whenever your documents change. The second happens **every time** someone asks something.

```text
document → chunk → embed → store  │  retrieve → augment → generate
        INDEXING (once, up front) │  QUERYING (per question)
```

**Indexing** (done ahead of time, whenever documents change):

1. **Document** — start with raw source: a PDF, a wiki page, a transcript, a DB row.
2. **Chunk** — split each document into smaller passages. A **chunk** is a bite-sized piece of text — a few sentences or a paragraph — small enough to be specific, big enough to stand on its own. (We chunk because we want to retrieve *the relevant paragraph*, not an entire 80-page manual.)
3. **Embed** — convert each chunk into an **embedding**: a list of numbers (a **vector**) that captures the chunk's *meaning*, so passages about similar ideas get similar numbers. This is what lets a computer search by meaning instead of by exact keywords — the entire subject of Part 2.
4. **Store** — save the vectors in a **vector store**, a database built to answer one question blazingly fast: "which stored vectors are most similar to *this* one?"

**Querying** (done per question):

5. **Retrieve** — embed the user's question into a vector too, then ask the store for the chunks whose vectors are closest. Closest-in-meaning ≈ most-relevant. These are the "right pages."
6. **Augment** — drop those retrieved chunks into the prompt alongside the question, with an instruction to answer from them.
7. **Generate** — send the assembled prompt to the LLM, which writes the grounded answer (ideally citing which chunk it used).

That's the entire arc: **document → chunk → embed → store → retrieve → augment → generate.** Don't worry about *how* any single stage works yet — each gets its own part later. For now, just keep the map in your head; the rest of the series is a guided tour of exactly these stations.

## When to reach for RAG (vs. a bigger prompt vs. fine-tuning)

RAG is not the only tool, and reaching for it reflexively is its own mistake. Here's how I decide between the three options people usually weigh.

- **Just use a bigger prompt** — paste the relevant text in by hand — when the knowledge is *tiny and known in advance*: a single policy, one short doc, a fixed style guide. If you can comfortably fit the source in the prompt and you already know which source it is, you don't need retrieval; you need copy-paste. Don't build a pipeline to solve a one-paragraph problem.
- **Reach for RAG** when the knowledge is *large, changing, private, or you don't know in advance which piece you'll need*: company wikis, product docs, support histories, contracts — anything that updates often or is too big to paste. Add, change, or remove a document and the system reflects it on the next question: no retraining, just re-indexing the part that changed.
- **Consider fine-tuning** — actually adjusting the model's weights on your own examples — when you need to change the model's *behavior, format, or style* rather than its *facts*. Fine-tuning is great at "always respond in this tone / this JSON shape" and poor at "know this fact," because facts baked into weights are expensive to update and still can't be cited.

A useful one-liner: **fine-tuning changes how the model talks; RAG changes what it knows right now.** Many serious systems use both — fine-tune the *manner*, RAG the *matter* — but if your pain is *wrong-about-my-data*, start with RAG.

## Recap

- A vanilla LLM predicts *likely text*; it has **no built-in notion of truth, no clock, and no access to your private data** — hence hallucination, knowledge cutoff, no proprietary knowledge, and context-window limits.
- **RAG = Retrieval + Augmentation + Generation:** find the relevant pieces of your data, put them in the prompt, and let the model answer from that evidence instead of memory alone.
- The durable mental model is the **open-book exam**: RAG doesn't make the model smarter, it lets it look the answer up — and **retrieval quality decides everything**.
- The pipeline is one arc: **document → chunk → embed → store → retrieve → augment → generate**, split into a one-time *indexing* phase and a per-question *querying* phase.
- Choose deliberately: a **bigger prompt** for tiny known sources, **RAG** for large/changing/private knowledge, **fine-tuning** for style and behavior rather than facts.

---

### Next: Part 2 — Embeddings — the code starts there

We cracked open the word "embed" and ran past it. Next we slow down and answer the question the whole pipeline rests on: *how do you turn the meaning of a sentence into numbers, and why does that make search by meaning possible?* That's where the first runnable code lives.

📖 **Read the essay:** https://www.mefby.com/essays/embeddings